In [1]:

def encrypt_columnar(plaintext, key):
    # Standardise text: remove spaces and convert to uppercase
    plaintext = plaintext.replace(" ", "").upper()
    key = key.upper()
    
    num_cols = len(key)
    num_rows = (len(plaintext) + num_cols - 1) // num_cols
    
    # Pad the plaintext with 'X' to fill the grid completely
    padded_len = num_rows * num_cols
    plaintext += 'X' * (padded_len - len(plaintext))
    
    # Get column reading order by sorting key characters alphabetically
    # e.g., 'CAT' -> index order [1, 0, 2]
    key_order = sorted(list(range(num_cols)), key=lambda k: key[k])
    
    # Build the grid row by row
    grid = [plaintext[i:i+num_cols] for i in range(0, len(plaintext), num_cols)]
    
    # Read columns vertically based on the sorted key order
    ciphertext = ""
    for col_idx in key_order:
        for row in grid:
            ciphertext += row[col_idx]
            
    return ciphertext


def decrypt_columnar(ciphertext, key):
    key = key.upper()
    num_cols = len(key)
    num_rows = len(ciphertext) // num_cols
    
    # Get the column reading order sequence
    key_order = sorted(list(range(num_cols)), key=lambda k: key[k])
    
    # Create an empty grid array
    grid = [['' for _ in range(num_cols)] for _ in range(num_rows)]
    
    # Populate the grid vertically column by column using the key order
    cipher_idx = 0
    for col_idx in key_order:
        for row_idx in range(num_rows):
            grid[row_idx][col_idx] = ciphertext[cipher_idx]
            cipher_idx += 1
            
    # Read horizontally row by row to reconstruct the plaintext
    plaintext = "".join(["".join(row) for row in grid])
    return plaintext


# --- Example Execution ---
if __name__ == "__main__":
    message = "HELLOWORLD"
    secret_key = "CAT"
    
    # Encrypt
    encrypted_msg = encrypt_columnar(message, secret_key)
    print(f"Original Text:  {message}")
    print(f"Key:            {secret_key}")
    print(f"Ciphertext:     {encrypted_msg}")  
    
    # Decrypt
    decrypted_msg = decrypt_columnar(encrypted_msg, secret_key)
    print(f"Decrypted Text: {decrypted_msg}")

Original Text:  HELLOWORLD
Key:            CAT
Ciphertext:     EORXHLODLWLX
Decrypted Text: HELLOWORLDXX


b. Double Transposition


In [9]:
def _encrypt_single(plaintext, key):
    num_cols = len(key)
    num_rows = (len(plaintext) + num_cols - 1) // num_cols
    
    # Pad grid with 'X' to make it regular
    padded_len = num_rows * num_cols
    plaintext += 'X' * (padded_len - len(plaintext))
    
    key_order = sorted(list(range(num_cols)), key=lambda k: key[k])
    grid = [plaintext[i:i+num_cols] for i in range(0, len(plaintext), num_cols)]
    
    ciphertext = ""
    for col_idx in key_order:
        for row in grid:
            ciphertext += row[col_idx]

    return ciphertext


def _decrypt_single(ciphertext, key):
    num_cols = len(key)
    num_rows = len(ciphertext) // num_cols
    
    key_order = sorted(list(range(num_cols)), key=lambda k: key[k])
    grid = [['' for _ in range(num_cols)] for _ in range(num_rows)]
    
    cipher_idx = 0
    for col_idx in key_order:
        for row_idx in range(num_rows):
            grid[row_idx][col_idx] = ciphertext[cipher_idx]
            cipher_idx += 1
            
    return "".join(["".join(row) for row in grid])


# --- Double Transposition ---

def encrypt_double(plaintext, key1, key2):
    """Encrypts text using two successive keys."""
    # Standardise text format
    clean_text = plaintext.replace(" ", "").upper()
    k1, k2 = key1.upper(), key2.upper()
    
    # Round 1
    intermediate = _encrypt_single(clean_text, k1)
    print("Cipher 1(intermediate): ",intermediate)
    # Round 2
    final_cipher = _encrypt_single(intermediate, k2)
    
    return final_cipher


def decrypt_double(ciphertext, key1, key2):
    """Decrypts text by reversing keys in backward order."""
    k1, k2 = key1.upper(), key2.upper()
    
    # Reverse Round 2 first, then Reverse Round 1
    intermediate = _decrypt_single(ciphertext, k2)
    original_text = _decrypt_single(intermediate, k1)
    
    return original_text


# --- Execution Example ---
if __name__ == "__main__":
    message = "ATTACKATDAWN"
    first_key = "CAT"
    second_key = "DOGa"
    
    # Run Encryption
    
    print(f"Original Text:  {message}")
    print(f"Key 1:  {first_key}")
    print(f"Key 2:  {second_key}")
    cipher = encrypt_double(message, first_key, second_key)
    print(f"Ciphertext:     {cipher}")
   
    # Run Decryption
    decrypted = decrypt_double(cipher, first_key, second_key)
    print(f"Decrypted Text: {decrypted}")

Original Text:  ATTACKATDAWN
Key 1:  CAT
Key 2:  DOGa
Cipher 1(intermediate):  TCTWAAAATKDN
Ciphertext:     WANTATTADCAK
Decrypted Text: ATTACKATDAWN
